In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:


# 1) Convert numpy arrays to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32)

# If labels are shape (N,), make them (N,1) for regression
if y_train_t.ndim == 1:
    y_train_t = y_train_t.unsqueeze(1)
if y_test_t.ndim == 1:
    y_test_t = y_test_t.unsqueeze(1)





In [ ]:
# 2) Create TensorDatasets
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)





In [ ]:
# 3) Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)




In [ ]:

# 4) Inspect one batch
X_batch, y_batch = next(iter(train_loader))
print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)


In [ ]:

# 5) Display a few images
# If images are flattened, try to reshape them (adjust H, W, C if needed)
plt.figure(figsize=(10, 4))
for i in range(6):
    img = X_batch[i].detach().cpu()

    # Handle common formats:
    # (C,H,W) -> (H,W,C)
    if img.ndim == 3 and img.shape[0] in [1, 3]:
        img = img.permute(1, 2, 0)

    plt.subplot(1, 6, i+1)
    plt.imshow(img.squeeze(), cmap="gray" if img.ndim == 2 or (img.ndim==3 and img.shape[-1]==1) else None)
    plt.axis("off")
    plt.title(int(y_batch[i].item()))
plt.tight_layout()
plt.show()

In [ ]:
# 1) Model class with 4 linear layers
class AgeRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # Flatten if input is image-like
        x = x.view(x.size(0), -1)
        return self.net(x)



In [ ]:
# 2) Training loop function
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(Xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.size(0)

    return total_loss / len(loader.dataset)



In [ ]:
# 3) Validation loop function
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        preds = model(Xb)
        loss = criterion(preds, yb)
        total_loss += loss.item() * Xb.size(0)

    return total_loss / len(loader.dataset)



In [ ]:
# 4) Define device, model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# infer input_dim from one batch
X_batch, _ = next(iter(train_loader))
input_dim = X_batch.view(X_batch.size(0), -1).shape[1]

model = AgeRegressor(input_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)



In [ ]:
# 5) Train for 20 epochs and track losses
train_losses, val_losses = [], []

for epoch in range(1, 21):
    tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss = validate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch {epoch:02d} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f}")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2 (Bonus): Write your code here: